# PastPOV — Kaggle production notebook

**Before running (right panel → Session options):**
1. Accelerator → **GPU T4 x2** (not P100)
2. **Internet → On** (needs phone verification on your Kaggle account)
3. Run cells top to bottom. First run: keep `MODE = "smoke"` in Cell 5. When it succeeds, set `MODE = "full"`, then use **Save Version → Save & Run All (Commit)** so it runs in the background and you can close the tab.


## Cell 1 — check the GPU and create folders

In [ ]:
import os
os.makedirs('/tmp/pastpov/code/episodes', exist_ok=True)
os.makedirs('/kaggle/working/PastPOV', exist_ok=True)
!nvidia-smi -L
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Cell 2 — install Wan2GP + Piper (≈10–20 min, once per session)
If this cell fails, copy the last ~25 lines and send them to Claude.

In [ ]:
%%bash
set -e
mkdir -p /tmp/pastpov && cd /tmp/pastpov
pip -q install uv

# ---- settings you may need to change if install fails ----
TORCH_SPEC="torch==2.8.0 torchvision torchaudio"
TORCH_INDEX="https://download.pytorch.org/whl/cu126"
# -----------------------------------------------------------

[ -d Wan2GP ] || git clone --depth 1 https://github.com/deepbeepmeep/Wan2GP.git
[ -f env/bin/python ] || uv venv env --python 3.10
uv pip install --python env/bin/python $TORCH_SPEC --index-url $TORCH_INDEX
uv pip install --python env/bin/python -r Wan2GP/requirements.txt

[ -f voice/bin/python ] || uv venv voice --python 3.11
uv pip install --python voice/bin/python "piper-tts>=1.3,<2"

env/bin/python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.device_count(), 'GPU(s)')"
voice/bin/python -c "import piper; print('piper OK')"
echo INSTALL_DONE


## Cell 3 — engine (don't edit)

In [ ]:
%%writefile /tmp/pastpov/code/pastpov_engine.py
#!/usr/bin/env python3
"""PastPOV production engine (Kaggle / any Linux GPU box).

Usage:
    python pastpov_engine.py [--smoke] episode1.json [episode2.json ...]

Per episode it: synthesizes the narration (Piper), generates one Wan2GP clip per
scene, edits clip + voice + synthesized FX, burns captions, and writes the result to
OUT/<episode id>/final and OUT/ready. Finished clips are cached and reused.

Environment variables (all optional):
    PASTPOV_WORK       env + Wan2GP + models folder      (default /tmp/pastpov)
    PASTPOV_OUT        saved outputs folder              (default /kaggle/working/PastPOV)
    PASTPOV_GPUS       1 or 2 parallel GPU workers       (default 1)
    PASTPOV_PROFILE    WanGP memory profile 1-5          (default 4)
    PASTPOV_MAX_HOURS  stop starting new clips after N h (default 8)
    PASTPOV_INTERP     none | blend | mci  (30fps smoothing, default blend)
    PASTPOV_RESUME     folder of a previous run's outputs to reuse clips from
"""

from pathlib import Path
import argparse
import array
import concurrent.futures
import hashlib
import json
import math
import os
import queue
import random
import re
import shutil
import subprocess
import sys
import textwrap
import threading
import time
import traceback
import urllib.request
import wave

VERSION = "pastpov-engine-2.0"
RENDER_VERSION = "render-2"

WORK = Path(os.environ.get("PASTPOV_WORK", "/tmp/pastpov"))
OUT = Path(os.environ.get("PASTPOV_OUT", "/kaggle/working/PastPOV"))
GPUS = max(1, int(os.environ.get("PASTPOV_GPUS", "1")))
PROFILE = os.environ.get("PASTPOV_PROFILE", "4")
INTERP = os.environ.get("PASTPOV_INTERP", "blend")
MAX_HOURS = float(os.environ.get("PASTPOV_MAX_HOURS", "8"))
RESUME = os.environ.get("PASTPOV_RESUME", "")

GEN_PY = WORK / "env" / "bin" / "python"
VOICE_PY = WORK / "voice" / "bin" / "python"
REPO = WORK / "Wan2GP"

MODEL = "t2v_1.3B"
RESOLUTION = "432x768"
STEPS = 25
FRAMES = 81          # Wan 1.3B is trained around 81 frames (~5 s at 16 fps)
MIN_SCENE_SECONDS = 5.0
VOICE_NAME = "en_US-ljspeech-high"
VOICE_URL = "https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/ljspeech/high/"
AUDIO_POST = "highpass=f=65,loudnorm=I=-16:TP=-1.5:LRA=9"

DEFAULT_STYLE = (
    "Photorealistic live-action historical reconstruction. "
    "Human eye-level first-person viewpoint. Realistic materials and natural restrained lighting. "
    "Single continuous shot. "
)
DEFAULT_NEGATIVE = (
    "cartoon, illustration, animation, CGI, video game, plastic textures, "
    "oversaturated orange, modern clothing, cars, modern objects, electrical wires, "
    "ruined tourist site, text, watermark, hands in foreground, distorted anatomy, "
    "close-up faces, blood, gore, bodies, corpses, lava river, abrupt cuts"
)

START = time.time()
DEADLINE = START + MAX_HOURS * 3600
LOG = None
LOG_LOCK = threading.Lock()


# ----------------------------------------------------------------- helpers
def write_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")
    temporary.replace(path)


def sha(value):
    blob = json.dumps(value, sort_keys=True, ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(blob).hexdigest()[:10]


def log_line(text):
    if LOG:
        with LOG_LOCK:
            with LOG.open("a", encoding="utf-8") as output:
                output.write(text if text.endswith("\n") else text + "\n")


def say(message):
    print(message, flush=True)
    log_line(message)


def run(args, cwd=None, env=None, quiet=False, tag="", stdin_text=None):
    """Run a subprocess, stream its output, raise with the log tail on failure."""
    import signal
    child_env = os.environ.copy()
    child_env.update({"MPLBACKEND": "Agg", "PYTHONUNBUFFERED": "1"})
    if env:
        child_env.update(env)
    process = subprocess.Popen(
        [str(a) for a in args], cwd=cwd, env=child_env,
        stdin=subprocess.PIPE if stdin_text is not None else subprocess.DEVNULL,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        errors="replace", bufsize=1, start_new_session=True,
    )
    tail = []
    prefix = f"[{tag}] " if tag else ""
    try:
        if stdin_text is not None:
            process.stdin.write(stdin_text)
            process.stdin.close()
        for line in process.stdout:
            tail.append(line.rstrip())
            tail = tail[-25:]
            if not quiet:
                print(prefix + line, end="", flush=True)
            log_line(prefix + line)
        code = process.wait()
        if code:
            raise RuntimeError("توقف التنفيذ. أرسل هذه الأسطر:\n" + "\n".join(tail))
    except BaseException:
        if process.poll() is None:
            os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
        raise


def media_info(path):
    result = subprocess.run([
        "ffprobe", "-v", "error", "-show_entries",
        "format=duration:stream=codec_type,width,height,r_frame_rate",
        "-of", "json", str(path),
    ], capture_output=True, text=True, check=True)
    return json.loads(result.stdout)


def duration(path):
    return float(media_info(path)["format"]["duration"])


def valid_video(path):
    path = Path(path)
    try:
        if not path.is_file() or path.stat().st_size < 1000:
            return False
        info = media_info(path)
        return float(info["format"]["duration"]) > 0.5 and any(
            s.get("codec_type") == "video" for s in info["streams"])
    except (OSError, ValueError, KeyError, subprocess.CalledProcessError):
        return False


def download(url, path):
    path = Path(path)
    if path.is_file() and path.stat().st_size > 100:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".download")
    for attempt in range(3):
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "PastPOV/2.0"})
            with urllib.request.urlopen(request, timeout=120) as response, temporary.open("wb") as out:
                shutil.copyfileobj(response, out)
            if temporary.stat().st_size < 100:
                raise RuntimeError("downloaded file is incomplete: " + url)
            temporary.replace(path)
            return
        except Exception:
            if attempt == 2:
                raise
            time.sleep(2)


# ------------------------------------------------------------------- audio
def sound_effect(path, seconds, intensity, seed):
    """Original low-volume rumble + wind, audible on phone speakers (70/105 Hz + 200-1600 Hz)."""
    sample_rate = 22050
    rng = random.Random(seed)
    k = max(0.0, min(1.0, intensity / 0.025))
    rumble_amp = 0.02 + 0.07 * k
    wind_amp = 0.004 + 0.035 * k
    samples = array.array("h")
    low = 0.0
    for index in range(math.ceil(seconds * sample_rate)):
        t = index / sample_rate
        low = 0.55 * low + 0.45 * rng.uniform(-1, 1)
        envelope = min(1.0, t / 0.4, max(0.0, (seconds - t) / 0.5))
        rumble = (0.6 * math.sin(2 * math.pi * 70 * t) + 0.4 * math.sin(2 * math.pi * 105 * t)) \
            * (0.8 + 0.2 * math.sin(2 * math.pi * 0.35 * t))
        wind = low * (0.6 + 0.4 * math.sin(2 * math.pi * 0.18 * t + seed))
        value = envelope * (rumble_amp * rumble + wind_amp * 2.0 * wind)
        samples.append(max(-32768, min(32767, round(value * 32767))))
    if sys.byteorder != "little":
        samples.byteswap()
    with wave.open(str(path), "wb") as output:
        output.setnchannels(1)
        output.setsampwidth(2)
        output.setframerate(sample_rate)
        output.writeframes(samples.tobytes())


# --------------------------------------------------------------- subtitles
def subtitle_chunks(text, max_words=5):
    """Split at sentence ends first, then into balanced pieces of at most max_words."""
    chunks = []
    for sentence in re.split(r"(?<=[.!?])\s+", text.strip()):
        words = sentence.split()
        if not words:
            continue
        pieces = math.ceil(len(words) / max_words)
        size = math.ceil(len(words) / pieces)
        chunks += [" ".join(words[i:i + size]) for i in range(0, len(words), size)]
    return chunks


def ass_time(seconds):
    value = round(seconds * 100)
    hours, value = divmod(value, 360000)
    minutes, value = divmod(value, 6000)
    secs, hundredths = divmod(value, 100)
    return f"{hours}:{minutes:02}:{secs:02}.{hundredths:02}"


def srt_time(seconds):
    value = round(seconds * 1000)
    hours, value = divmod(value, 3600000)
    minutes, value = divmod(value, 60000)
    secs, milliseconds = divmod(value, 1000)
    return f"{hours:02}:{minutes:02}:{secs:02},{milliseconds:03}"


def clean_ass(text):
    return str(text).replace("{", "(").replace("}", ")").replace("\n", " ")


def make_subtitles(plan, folder, ep):
    total = sum(item["duration"] for item in plan)
    header = """[Script Info]
ScriptType: v4.00+
PlayResX: 720
PlayResY: 1280
WrapStyle: 0
ScaledBorderAndShadow: yes

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Captions,DejaVu Sans,46,&H00FFFFFF,&H00FFFFFF,&H00101010,&H90000000,-1,0,0,0,100,100,0,0,1,3,1,2,48,48,255,1
Style: Title,DejaVu Sans,36,&H00FFFFFF,&H00FFFFFF,&H00101010,&H90000000,-1,0,0,0,100,100,0,0,1,2,1,8,40,40,72,1
Style: Label,DejaVu Sans,22,&H00FFFFFF,&H00FFFFFF,&H00101010,&H90000000,0,0,0,0,100,100,0,0,1,1,0,8,40,40,122,1
Style: Brand,DejaVu Sans,24,&H00D5E7F5,&H00FFFFFF,&H00101010,&H90000000,-1,0,0,0,100,100,0,0,1,1,0,1,40,40,62,1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""
    brand = clean_ass(ep.get("brand", "PAST POV"))
    title = clean_ass(ep.get("title_card", ep["id"]))
    label = clean_ass(ep.get("label", "AI HISTORICAL RECONSTRUCTION"))
    lines = [
        header,
        f"Dialogue: 1,{ass_time(0)},{ass_time(total)},Brand,,0,0,0,,{brand}\n",
        f"Dialogue: 1,{ass_time(0)},{ass_time(5)},Title,,0,0,0,,{title}\n",
        f"Dialogue: 1,{ass_time(0)},{ass_time(5)},Label,,0,0,0,,{label}\n",
    ]
    srt = []
    offset = 0.0
    index = 1
    for item in plan:
        chunks = subtitle_chunks(item["narration"])
        weights = [len(chunk.split()) for chunk in chunks]
        spoken = min(item["voice_duration"], item["duration"] - 0.2)
        cursor = offset
        for chunk, weight in zip(chunks, weights):
            end = cursor + spoken * weight / sum(weights)
            wrapped = "\n".join(textwrap.wrap(chunk, width=24, break_on_hyphens=False))
            ass_text = wrapped.replace("\n", r"\N").replace("{", "(").replace("}", ")")
            lines.append(f"Dialogue: 0,{ass_time(cursor)},{ass_time(end)},Captions,,0,0,0,,{ass_text}\n")
            srt.append(f"{index}\n{srt_time(cursor)} --> {srt_time(end)}\n{wrapped}\n")
            index += 1
            cursor = end
        offset += item["duration"]
    ass_path = Path(folder) / "captions.ass"
    srt_path = Path(folder) / "captions.srt"
    ass_path.write_text("".join(lines), encoding="utf-8")
    srt_path.write_text("\n".join(srt), encoding="utf-8")
    return ass_path, srt_path


# ----------------------------------------------------------------- editing
def video_chain(speed, seconds, last):
    chain = f"[0:v]setpts=PTS-STARTPTS,setpts={speed:.9f}*PTS,"
    if INTERP == "mci":
        chain += "minterpolate=fps=30:mi_mode=mci:mc_mode=aobmc:vsbmc=1,"
    elif INTERP == "blend":
        chain += "minterpolate=fps=30:mi_mode=blend,"
    chain += (
        "scale=720:1280:flags=lanczos,setsar=1,fps=30,"
        f"tpad=stop_mode=clone:stop_duration=0.2,trim=duration={seconds:.6f},"
        "format=yuv420p"
    )
    if last:
        chain += f",fade=t=out:st={max(0, seconds - 0.55):.6f}:d=0.55"
    return chain + "[v];"


def render_segment(raw, voice, ambience, target, seconds, last=False):
    temporary = target.with_name(target.stem + ".tmp.mp4")
    speed = seconds / duration(raw)
    audio_filter = (
        f"[1:a]aresample=48000,apad,atrim=duration={seconds:.6f},asetpts=PTS-STARTPTS[voice];"
        f"[2:a]aresample=48000,highpass=f=45,atrim=duration={seconds:.6f},asetpts=PTS-STARTPTS[fx];"
        "[voice][fx]amix=inputs=2:duration=longest:normalize=0,"
        "alimiter=limit=0.95:latency=1[a]"
    )
    run([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-i", raw, "-i", voice, "-i", ambience,
        "-filter_complex", video_chain(speed, seconds, last) + audio_filter,
        "-map", "[v]", "-map", "[a]", "-t", f"{seconds:.6f}",
        "-c:v", "libx264", "-preset", "fast", "-crf", "19", "-threads", "2",
        "-c:a", "aac", "-b:a", "160k", "-ar", "48000", "-ac", "2",
        "-movflags", "+faststart", temporary,
    ])
    if not valid_video(temporary):
        raise RuntimeError("فشل التحقق من مشهد المونتاج")
    temporary.replace(target)


def assemble(segments, folder, ass_path, episode_id):
    folder = Path(folder)
    concat_file = folder / "concat.txt"
    concat_file.write_text("".join(f"file '{p.as_posix()}'\n" for p in segments), encoding="utf-8")
    clean = folder / f"PastPOV_{episode_id}_clean.mp4"
    clean_temp = folder / f"PastPOV_{episode_id}_clean.tmp.mp4"
    run([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-f", "concat", "-safe", "0", "-i", concat_file,
        "-c", "copy", "-movflags", "+faststart", clean_temp,
    ])
    clean_temp.replace(clean)
    final = folder / f"PastPOV_{episode_id}_FINAL.mp4"
    temporary = folder / f"PastPOV_{episode_id}_FINAL.tmp.mp4"
    run([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", clean,
        "-vf", f"subtitles=filename='{Path(ass_path).as_posix()}'",
        "-c:v", "libx264", "-preset", "fast", "-crf", "19", "-threads", "2",
        "-pix_fmt", "yuv420p", "-c:a", "copy", "-movflags", "+faststart", temporary,
    ])
    if not valid_video(temporary):
        raise RuntimeError("الفيديو النهائي غير صالح")
    temporary.replace(final)
    return final, clean


# ------------------------------------------------------------------- setup
def ffmpeg_ok():
    try:
        out = subprocess.run(["ffmpeg", "-hide_banner", "-filters"],
                             capture_output=True, text=True).stdout
        subprocess.run(["ffprobe", "-version"], capture_output=True, check=True)
    except (FileNotFoundError, subprocess.CalledProcessError):
        return False
    return " subtitles " in out and " minterpolate " in out


def ensure_tools():
    if not ffmpeg_ok():
        say("تثبيت ffmpeg عبر apt ...")
        run(["apt-get", "update", "-qq"], quiet=True)
        run(["apt-get", "install", "-y", "-qq", "ffmpeg", "fonts-dejavu-core"], quiet=True)
        os.environ["PATH"] = "/usr/bin:" + os.environ["PATH"]
        if not ffmpeg_ok():
            raise RuntimeError("ffmpeg بلا فلتر subtitles/minterpolate؛ أرسل هذه الرسالة.")
    if not GEN_PY.is_file() or not (REPO / "wgp.py").is_file():
        raise RuntimeError("بيئة Wan2GP غير مثبتة. شغّل خلية التثبيت (Cell 2) أولًا.")
    if not VOICE_PY.is_file():
        raise RuntimeError("بيئة Piper غير مثبتة. شغّل خلية التثبيت (Cell 2) أولًا.")


def ensure_voice():
    voices = WORK / "voices"
    for suffix in (".onnx", ".onnx.json"):
        download(VOICE_URL + VOICE_NAME + suffix, voices / (VOICE_NAME + suffix))
    return voices / (VOICE_NAME + ".onnx")


def seed_from(previous, target_root):
    """Copy still-valid clips/audio/segments from an earlier run so they are not regenerated."""
    previous = Path(previous)
    if not previous.is_dir():
        say(f"RESUME folder not found: {previous}")
        return
    copied = 0
    for source in previous.rglob("*"):
        if source.is_file() and source.suffix in (".mp4", ".wav") and \
                source.parent.name in ("raw", "audio", "segments"):
            relative = source.relative_to(previous)
            destination = Path(target_root) / relative
            if not destination.exists():
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(source, destination)
                copied += 1
    say(f"استئناف: تم نسخ {copied} ملفًا محفوظًا من {previous}")


# ------------------------------------------------------------------- plan
def make_plan(ep, root, smoke):
    style = ep.get("style", DEFAULT_STYLE)
    negative = ep.get("negative", DEFAULT_NEGATIVE)
    frames = 33 if smoke else FRAMES
    steps = 10 if smoke else STEPS
    scenes = ep["scenes"][:1] if smoke else ep["scenes"]
    plan = []
    for index, scene in enumerate(scenes):
        seed = int(ep.get("seed_base", 7901)) + index
        vkey = sha({"v": VERSION, "model": MODEL, "res": RESOLUTION, "steps": steps,
                    "frames": frames, "seed": seed, "style": style,
                    "negative": negative, "visual": scene["visual"]})
        akey = sha({"voice": VOICE_NAME, "text": scene["narration"], "post": AUDIO_POST})
        name = scene["name"]
        plan.append({
            **scene, "index": index, "seed": seed, "vkey": vkey, "akey": akey,
            "raw": root / "raw" / f"{name}_v{vkey}.mp4",
            "voice": root / "audio" / f"{name}_a{akey}.wav",
            "job": {
                "model_type": MODEL, "prompt": style + scene["visual"],
                "negative_prompt": negative, "resolution": RESOLUTION,
                "video_length": frames, "num_inference_steps": steps,
                "seed": seed, "repeat_generation": 1,
            },
        })
    return plan


def prepare_voice(plan, voice_model):
    for item in plan:
        voice = item["voice"]
        if voice.is_file() and voice.stat().st_size > 1000:
            continue
        spoken = voice.with_name(voice.stem + "_unprocessed.wav")
        temporary = voice.with_name(voice.stem + ".tmp.wav")
        run([VOICE_PY, "-m", "piper", "-m", voice_model, "-f", spoken],
            quiet=True, stdin_text=item["narration"].strip() + "\n")
        run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", spoken,
             "-af", AUDIO_POST, "-ar", "48000", "-ac", "1", "-c:a", "pcm_s16le", temporary])
        temporary.replace(voice)
        spoken.unlink(missing_ok=True)
    for item in plan:
        item["voice_duration"] = duration(item["voice"])
        item["duration"] = math.ceil(max(MIN_SCENE_SECONDS, item["voice_duration"] + 0.45) * 30) / 30
        item["segment_key"] = sha({"v": item["vkey"], "a": item["akey"], "d": item["duration"],
                                   "r": item.get("rumble", 0.01), "interp": INTERP,
                                   "render": RENDER_VERSION})


# -------------------------------------------------------------- generation
def wgp_command(settings, outdir):
    return [GEN_PY, "-u", "wgp.py", "--process", settings, "--output-dir", outdir,
            "--attention", "sdpa", "--profile", PROFILE, "--fp16"]


def generate_clip(item, gpu):
    job_dir = WORK / "gen" / item["raw"].stem
    shutil.rmtree(job_dir, ignore_errors=True)
    (job_dir / "out").mkdir(parents=True)
    settings = job_dir / "job.json"
    write_json(settings, item["job"])
    started = time.time()
    run(wgp_command(settings, job_dir / "out"), cwd=REPO,
        env={"CUDA_VISIBLE_DEVICES": str(gpu)}, tag=f"gpu{gpu}:{item['name']}", quiet=GPUS > 1)
    produced = sorted((job_dir / "out").rglob("*.mp4"), key=lambda p: p.stat().st_mtime)
    if not produced:
        raise RuntimeError(f"Wan2GP لم ينتج ملفًا للمشهد {item['name']}")
    item["raw"].parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(produced[-1]), str(item["raw"]))
    shutil.rmtree(job_dir, ignore_errors=True)
    if not valid_video(item["raw"]):
        item["raw"].unlink(missing_ok=True)
        raise RuntimeError(f"الملف الناتج غير صالح: {item['name']}")
    return round(time.time() - started, 1)


def generate_clips(plan, episode_id):
    pending = []
    for item in plan:
        if valid_video(item["raw"]):
            say(f"محفوظ مسبقًا: {item['name']}")
            continue
        item["raw"].unlink(missing_ok=True)
        pending.append(item)
    if not pending:
        return
    say(f"سيتم توليد {len(pending)} مشهد (GPU workers: {GPUS})")
    first = pending[0]
    probe = WORK / "gen" / "dryrun"
    shutil.rmtree(probe, ignore_errors=True)
    (probe / "out").mkdir(parents=True)
    write_json(probe / "job.json", first["job"])
    run(wgp_command(probe / "job.json", probe / "out") + ["--dry-run"], cwd=REPO,
        env={"CUDA_VISIBLE_DEVICES": "0"})

    free_gpus = queue.Queue()
    for g in range(GPUS):
        free_gpus.put(g)

    def work(item):
        if time.time() > DEADLINE:
            return item, None, "skipped (time budget)"
        gpu = free_gpus.get()
        try:
            seconds = generate_clip(item, gpu)
            return item, seconds, None
        except Exception as error:
            return item, None, str(error)
        finally:
            free_gpus.put(gpu)

    timings_path = OUT / "timings.json"
    timings = json.loads(timings_path.read_text()) if timings_path.is_file() else []
    with concurrent.futures.ThreadPoolExecutor(max_workers=GPUS) as pool:
        for item, seconds, error in pool.map(work, pending):
            if error:
                say(f"✗ {item['name']}: {error.splitlines()[0] if error else error}")
                log_line(error)
            else:
                say(f"✓ {item['name']} — {seconds} ثانية")
                timings.append({"episode": episode_id, "scene": item["name"], "seconds": seconds,
                                "frames": item["job"]["video_length"],
                                "steps": item["job"]["num_inference_steps"], "gpus": GPUS})
                write_json(timings_path, timings)


# ----------------------------------------------------------------- episode
def produce_episode(path, voice_model, smoke):
    ep = json.loads(Path(path).read_text(encoding="utf-8"))
    episode_id = ep["id"]
    root = OUT / (episode_id + ("_SMOKE" if smoke else ""))
    for child in ("raw", "audio", "segments", "final", "logs"):
        (root / child).mkdir(parents=True, exist_ok=True)
    say(f"\n=== {episode_id} {'(smoke test)' if smoke else ''} ===")
    if RESUME:
        seed_from(RESUME, root.parent)

    plan = make_plan(ep, root, smoke)
    say("1/4 — التعليق الصوتي")
    prepare_voice(plan, voice_model)
    for item in plan:
        stretch = item["duration"] / (FRAMES / 16)
        if not smoke and stretch > 1.45:
            say(f"⚠ {item['name']}: الفيديو سيُبطَّأ ×{stretch:.2f}. الأفضل تقصير التعليق (~14 كلمة).")
    (root / "narration.txt").write_text("\n\n".join(s["narration"] for s in plan), encoding="utf-8")

    say("2/4 — توليد المشاهد (Wan2GP)")
    generate_clips(plan, episode_id)
    missing = [i["name"] for i in plan if not valid_video(i["raw"])]
    if missing:
        raise RuntimeError("مشاهد لم تكتمل: " + ", ".join(missing) +
                           ". أعد التشغيل للاستئناف (المشاهد المكتملة محفوظة).")

    say("3/4 — المونتاج")
    segments = []
    for i, item in enumerate(plan):
        target = root / "segments" / f"{item['name']}_s{item['segment_key']}.mp4"
        if not valid_video(target):
            fx = root / "audio" / f"{item['name']}_{item['segment_key']}_fx.wav"
            sound_effect(fx, item["duration"], item.get("rumble", 0.01), 800 + i)
            render_segment(item["raw"], item["voice"], fx, target, item["duration"],
                           last=i == len(plan) - 1)
        segments.append(target)
        say(f"تم مونتاج: {item['name']}")
    if smoke:
        say(f"✅ Smoke OK — ملف تجريبي: {segments[0]}")
        return {"episode": episode_id, "status": "smoke_ok", "file": str(segments[0])}

    say("4/4 — الدمج والترجمة")
    ass_path, srt_path = make_subtitles(plan, root / "final", ep)
    final, clean = assemble(segments, root / "final", ass_path, episode_id)
    info = media_info(final)
    expected = sum(i["duration"] for i in plan)
    if abs(float(info["format"]["duration"]) - expected) > 0.6:
        raise RuntimeError("مدة الفيديو لا تطابق خطة المونتاج؛ أرسل هذه الرسالة.")
    if not any(s.get("codec_type") == "audio" for s in info["streams"]):
        raise RuntimeError("الفيديو النهائي يفتقد الصوت")
    ready = OUT / "ready"
    ready.mkdir(parents=True, exist_ok=True)
    shutil.copy2(final, ready / f"{episode_id}.mp4")
    shutil.copy2(srt_path, ready / f"{episode_id}.srt")
    if ep.get("caption"):
        (ready / f"{episode_id}_caption.txt").write_text(ep["caption"], encoding="utf-8")
    write_json(root / "final" / "verification.json", {
        "episode": episode_id, "expected_duration": expected, "actual": info,
        "model": MODEL, "resolution": RESOLUTION, "steps": STEPS, "frames": FRAMES,
        "voice": VOICE_NAME, "engine": VERSION,
    })
    say(f"FINAL_OK ✅ {ready / (episode_id + '.mp4')}  ({round(duration(final), 1)} ثانية)")
    return {"episode": episode_id, "status": "done", "file": str(ready / f"{episode_id}.mp4")}


def main():
    global LOG
    parser = argparse.ArgumentParser()
    parser.add_argument("episodes", nargs="+")
    parser.add_argument("--smoke", action="store_true",
                        help="test: first scene only, 33 frames, 10 steps")
    args = parser.parse_args()
    OUT.mkdir(parents=True, exist_ok=True)
    WORK.mkdir(parents=True, exist_ok=True)
    LOG = OUT / "run_log.txt"
    os.environ["MPLBACKEND"] = "Agg"
    ensure_tools()
    voice_model = ensure_voice()
    results = []
    for path in args.episodes:
        try:
            results.append(produce_episode(path, voice_model, args.smoke))
        except Exception as error:
            say(f"✗ فشل {path}: {error}")
            log_line(traceback.format_exc())
            results.append({"episode": str(path), "status": "failed", "error": str(error)[:2000]})
    write_json(OUT / "run_summary.json", {"engine": VERSION, "results": results,
                                          "hours": round((time.time() - START) / 3600, 2)})
    say("\nالملخص:")
    for r in results:
        say(f" - {r['episode']}: {r['status']}")
    # Exit 0 on purpose so a Kaggle commit keeps the outputs even if one episode failed.


if __name__ == "__main__":
    main()


## Cell 4 — episodes
One JSON per video. To add a new video: copy the template cell, change the content, and add its path to `EPISODES` in Cell 5.

In [ ]:
%%writefile /tmp/pastpov/code/episodes/pompeii_ep01.json
{
  "id": "Pompeii_EP01",
  "title_card": "POV: POMPEII — AD 79",
  "brand": "PAST POV",
  "label": "AI HISTORICAL RECONSTRUCTION",
  "seed_base": 7901,
  "style": "Photorealistic live-action historical reconstruction of Pompeii in AD 79. Human eye-level first-person viewpoint. Realistic materials and natural restrained lighting. Single continuous shot. ",
  "caption": "POV: You wake up in Pompeii in AD 79. Would you run at the first tremor, or wait? 🌋\n\nAI historical reconstruction. Original script and visuals.\n\n#PastPOV #Pompeii #History #POV #AncientRome",
  "scenes": [
    {
      "name": "01_wake_up",
      "narration": "You wake up in Pompeii. It's the year seventy-nine. Above the city, Vesuvius looks strangely quiet.",
      "visual": "Camera slowly moving out of a small Roman room into an empty sunlit courtyard. Intact painted plaster walls, terracotta roof tiles, linen curtain moving in a breeze. A distant mountain is visible beyond the roofs.",
      "rumble": 0.002
    },
    {
      "name": "02_bakery",
      "narration": "You pass a bakery. Bread is cooling on the counter. Then the ground begins to shake.",
      "visual": "Camera gliding past an ancient Roman bakery in a living city. Round scored bread rests on a wooden counter beside a stone oven. Painted plaster walls, empty street. A subtle brief camera vibration near the end.",
      "rumble": 0.006
    },
    {
      "name": "03_eruption",
      "narration": "You look up. A towering column of ash rises above the mountain. People stop to stare.",
      "visual": "Camera gently tilts upward from intact Roman terracotta rooftops toward Mount Vesuvius beyond the city. A tall gray volcanic ash column rises and spreads across the blue sky. Slow natural cloud motion.",
      "rumble": 0.013
    },
    {
      "name": "04_pumice",
      "narration": "Ash drifts down. Soon, pale pumice stones strike the rooftops. This is no ordinary storm.",
      "visual": "View from beneath the shelter of an intact Roman doorway onto an empty narrow street. Light gray ash drifts across the street and small pale pumice stones fall onto stone paving and terracotta roofs. Painted plaster walls. Daylight begins to dim. Slow forward camera motion.",
      "rumble": 0.016
    },
    {
      "name": "05_escape",
      "narration": "You turn toward the gate. Some people flee. Others hurry back for the things they cannot leave.",
      "visual": "Camera moving along a Roman stone street toward a distant city gate. Two small distant figures in simple linen Roman tunics, seen only from behind, hurry away from the camera. Intact plastered buildings and a gray ash-filled sky.",
      "rumble": 0.014
    },
    {
      "name": "06_hours_later",
      "narration": "Hours pass. The sky darkens, and fallen pumice piles up. Every step toward safety becomes harder.",
      "visual": "Camera moving slowly through a dark empty ancient Roman street. Pale pumice stones cover the paving. Ash falls through dim torchlight under an overcast sky. Intact plastered houses and heavy dusty air. Continuous slow forward motion.",
      "rumble": 0.018
    },
    {
      "name": "07_hot_currents",
      "narration": "By the next morning, hot currents of gas and ash sweep toward those still in the city.",
      "visual": "View from inside a Roman city gateway facing the distant volcanic slopes. A dense gray ground-hugging ash cloud advances from the mountain toward the city. Terracotta roofs and intact plastered walls. Camera slowly retreats beneath the gateway.",
      "rumble": 0.025
    },
    {
      "name": "08_last_question",
      "narration": "Would you run at the first tremor, or wait until the sky turns black? Step into history.",
      "visual": "Camera looking along an empty ancient Roman street through thick gray volcanic haze. Ash settles on stone paving and intact plastered buildings. Slow natural ash motion, subdued light, solemn cinematic atmosphere.",
      "rumble": 0.012
    }
  ]
}

In [ ]:
%%writefile /tmp/pastpov/code/episodes/TEMPLATE.json
{
  "id": "NEXT_EP02",
  "title_card": "POV: PLACE — YEAR",
  "brand": "PAST POV",
  "label": "AI HISTORICAL RECONSTRUCTION",
  "seed_base": 8001,
  "style": "Photorealistic live-action historical reconstruction of ... Human eye-level first-person viewpoint. Realistic materials and natural restrained lighting. Single continuous shot. ",
  "caption": "POV: ... \n\nAI historical reconstruction. Original script and visuals.\n\n#PastPOV #History #POV",
  "scenes": [
    {
      "name": "01_scene",
      "narration": "One or two short sentences, about 12-14 words (about 5 seconds spoken).",
      "visual": "Describe ONLY what the camera sees, in positive words. Do not write 'no ...' here; unwanted things go in the negative prompt.",
      "rumble": 0.01
    }
  ]
}

## Cell 5 — run
- `MODE = "smoke"`: first scene only, 33 frames, 10 steps (fast test, tells you the real speed on this GPU).
- `MODE = "full"`: the whole episode(s).
- `GPUS = "1"` first; try `"2"` after one successful run (uses both T4s in parallel; watch for out-of-memory).

In [ ]:
import os, subprocess, sys

MODE = "smoke"        # "smoke" first, then "full"
GPUS = "1"            # "1" or "2"
EPISODES = [
    "/tmp/pastpov/code/episodes/pompeii_ep01.json",
    # "/tmp/pastpov/code/episodes/another_episode.json",
]
RESUME = ""           # optional: path of a previous run's output folder, e.g. /kaggle/input/<notebook>/PastPOV

os.environ.update(PASTPOV_GPUS=GPUS, PASTPOV_RESUME=RESUME, PASTPOV_MAX_HOURS="8",
                  PASTPOV_PROFILE="4", PASTPOV_INTERP="blend")
cmd = [sys.executable, "/tmp/pastpov/code/pastpov_engine.py"] + (["--smoke"] if MODE == "smoke" else []) + EPISODES
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
process.wait()
print("exit code:", process.returncode)


## Cell 6 — results
Finished videos are in `/kaggle/working/PastPOV/ready/` (download them from the notebook's **Output** tab).

In [ ]:
import json, pathlib
out = pathlib.Path("/kaggle/working/PastPOV")
for f in sorted((out / "ready").glob("*")):
    print(f"{f.stat().st_size/1e6:7.1f} MB  {f.name}")
t = out / "timings.json"
if t.exists():
    rows = json.loads(t.read_text())
    avg = sum(r["seconds"] for r in rows) / len(rows)
    print(f"\naverage seconds per clip on this GPU: {avg:.0f}  ({len(rows)} clips measured)")
s = out / "run_summary.json"
if s.exists(): print(json.loads(s.read_text()))
